# AIME-Con Workshop 
*Text Classification with Large Language Models: Pipelines, Fine-tuning, and Measurement Validity*





### Step B) Environment Set-Up

Local option: 

Create a virtual environment. The workbook was built for Python 3.12.3. Download it from [python.org](https://www.python.org/downloads/release/python-3123/)

In [ ]:
# local option: set up a virtual environment
# colab uses Python 3.12.3
!python3.12.3 -m venv.venv


Install modules from a requirements file. 

In [ ]:
!pip3 install -r requirements.txt

Set `USER` below. Either 
- `USER = "colab"` for working on Google Colab or
- `USER = "local` for working on your local computer.

In [ ]:
# import packages used in this script
import os

from pathlib import Path
import pandas as pd
from openai import OpenAI

# Non-Local Inference
Non-local calls are one way to interact with large language models with Python. 

You send a request over the internet to a provider's servers (e.g., OpenAI, Anthropic, Google) and receive a generated response back. 

### Step C) Set API Keys
Save your OpenAI API key. 

If you are running in Google CoLab

If you are running locally, save your key as a variable `OPENAI_API_KEY = ""` in a file named .env (see template in 00_secrets.txt).


In [ ]:
# check that the api key got loaded in the .env
# if loaded, prints the key
# if not loaded, prints ERROR
key = "OPENAI_API_KEY"
print(os.environ.get(key, f"ERROR: Variable {key} Not Found"))

# ⚙️ Model Settings

**System Prompt**

System prompts (also called System messages) are persistent instructions for how the model should behave. Once set, they guide every subsequent interaction.

how do these work in an api context? do they persist with the token?

**User Prompt**

User


In [ ]:
# ---- GPT ----

# initialize model
openai = OpenAI()

# format prompt
prompt = [
    #{"role": "system", "content": CONTEXT},
    {"role": "You are a researcher in educational psychology.", "content": "What is a dialogic prompt?"}
  ]

**Model**

Can select from different models at:
https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create 

**Temperature**

A parameter that controls the randomness of text generated by LLMs during inference. Each token is assigned a probability of occurance based on the prompt and the tokens that come before it. Temperature modifies this probability distribution such that at higher temperatures increase the likelihood of selecting less probable tokens.


**Top P**

An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with `top_p` probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.

**Token Limits**

The maximum number of tokens output by the model

# model settings
prompt_gpt = openai.chat.completions.create(
    model = gpt-5.6-terra, 
    messages = prompt,
    temperature = 0.1
    )

response = prompt_gpt.choices[0].message.content

print(prompt_gpt)

# Classify a Dialogic Prompt


construct prompt using f-string

In [ ]:
CASE = "Okay, and what would you do?"

PROMPT = f"Classify the following utterance as either a 1 = dialogic prompt or 0 = not a dialogic prompt. A dialogic prompt is defined as an utterance that implies, encourages, requests, or expects a new speaker (or multiple new speakers) to make a verbal contribution. Here is the utterance: \"{CASE}\" Return only 0 or 1."

define model parameters

In [ ]:
GPT_MODEL = "gpt-5.6-terra" # https://developers.openai.com/api/docs/models/all
CONTEXT = "You are an educational researcher" # SYSTEM PROMPT
TOKENS = 100
TEMPERATURE = 0.1

# cost per million tokens
IN_RATE = 2.00
OUT_RATE = 12.00

In [ ]:
# ---- GPT ----
# get api key: https://platform.openai.com/api-keys

# check that the api key got loaded in the .env
# if loaded, prints the key
# if not loaded, prints ERROR
key = "OPENAI_API_KEY"
print(os.environ.get(key, f"ERROR: Variable {key} Not Found"))

# initialize model
openai = OpenAI()

# format prompt
prompt = [
    #{"role": "system", "content": CONTEXT},
    {"role": "user", "content": PROMPT}
  ]

# model settings
# https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create
prompt_gpt = openai.chat.completions.create(
    model = GPT_MODEL, 
    messages = prompt,
    temperature = TEMPERATURE # range = 0-2
    )

response = prompt_gpt.choices[0].message.content

print(f"{GPT_MODEL} classified this as a {response}")


# 📋 Formatting Output
json


tokens -- classification + explanation 
show one case with format and explanation but then iterate through just using 
can use this to look at a sample, for example of cases where there was disagreement between the human raters and the llm maybe call back to this after iterating through?

In [ ]:
# cases used in ppt examples (df row will be csv row - 2 -- index and column headings)
# 136 = prompt
# 207 = prompt
# 223 = not a prompt
# 3558 = not a prompt
CASE = "Okay, and what would you do?"

PROMPT = f"Classify the following utterance as either a 1 = dialogic prompt or 0 = not a dialogic prompt. A dialogic prompt is defined as an utterance that implies, encourages, requests, or expects a new speaker (or multiple new speakers) to make a verbal contribution. Here is the utterance: \"{CASE}\" Return only 0 or 1."

Another way to do the above:

In [ ]:

FOCUS_DIR = Path("/Users/brittneyhernandez/Library/CloudStorage/OneDrive-UniversityofConnecticut/focus/project focus/focus_main")
AIMECON_DIR = Path("/Users/brittneyhernandez/Library/CloudStorage/OneDrive-UniversityofConnecticut/AIME-con")

FOCUS_DATA_DIR = FOCUS_DIR / "data/prompt_codes/cgi"
AIMECON_DATA_DIR = AIMECON_DIR / "data"

TRAIN_FILE = AIMECON_DATA_DIR / "cgi_train.xlsx"
VAL_FILE = AIMECON_DATA_DIR / "cgi_validate.xlsx"
TEST_FILE = AIMECON_DATA_DIR / "cgi_test.xlsx"

# ---- get prompt codebook ----
path_to_prompts = AIMECON_DIR / "data_management" / "prompt_codebook.xlsx"
prompts = pd.read_excel(path_to_prompts)

# ---- get data ----
path_to_data = TRAIN_FILE
df = pd.read_excel(path_to_data)

CASE = df.loc[134, "text"]

PROMPT = (prompts.loc[prompts.id == "Coding", "prompt"].item() + 
          prompts.loc[prompts.id == "Construct", "prompt"].item() +
          prompts.loc[prompts.id == "Prompt1", "prompt"].item() +
          f"\"{CASE}\"" + 
          prompts.loc[prompts.id == "Format", "prompt"].item()
          )

Run the loop on 30 cases each

## What did human raters classify the prompt as?

In [ ]:
print(df.loc[134, "code_human"])

# Loops

iterating through so we can automatically combine the prompt template with each case
then 

# ⏭️ Retries & Rate Limits
# 💰 Token Usage & Cost
Estimate cost based on document metadata
utterances, need to get total n tokens

In [ ]:
# https://developers.openai.com/api/docs/models/gpt-4o
in_rate = 2.50
out_rate = 10.00

input_cost = prompt_gpt.usage.prompt_tokens / 1_000_000 * in_rate
output_cost = prompt_gpt.usage.completion_tokens / 1_000_000 * out_rate

print(f"Prompt Tokens = {prompt_gpt.usage.prompt_tokens} (${input_cost})")
print(f"Completion Tokens = {prompt_gpt.usage.completion_tokens} (${output_cost})")

# Performance metrics
uncertainty